In [30]:
path_to_pdf=r"D:\newthings\StructRAG\centOSdepolyment.pdf"


In [ ]:
from unstructured.partition.pdf import partition_pdf
import hashlib 
import re




In [32]:
elements = partition_pdf(
    filename=path_to_pdf,
    strategy="fast",
    infer_table_structure=False
)



No languages specified, defaulting to English.


In [33]:
structured_data = []

for el in elements:

    text = str(el).strip()

    if not text:
        continue

    metadata = el.metadata.to_dict()

    structured_data.append({
        "type": el.category,
        "text": text,
        "page": metadata.get("page_number"),
        "filename": metadata.get("filename"),
        "coordinates": metadata.get("coordinates")
    })



In [34]:
print(structured_data)

[{'type': 'Title', 'text': 'Centos OS deployment of Django', 'page': 1, 'filename': 'centOSdepolyment.pdf', 'coordinates': {'points': ((90.0, 79.81145000000004), (90.0, 137.54144999999994), (351.06750000000005, 137.54144999999994), (351.06750000000005, 79.81145000000004)), 'system': 'PixelSpace', 'layout_width': 595.3, 'layout_height': 841.9}}, {'type': 'Title', 'text': 'and Apache', 'page': 1, 'filename': 'centOSdepolyment.pdf', 'coordinates': {'points': ((254.4, 79.81145000000004), (254.4, 105.86144999999999), (405.8615000000001, 105.86144999999999), (405.8615000000001, 79.81145000000004)), 'system': 'PixelSpace', 'layout_width': 595.3, 'layout_height': 841.9}}, {'type': 'NarrativeText', 'text': 'Step-1 Update the system and install Apache and mod_wsgi:', 'page': 1, 'filename': 'centOSdepolyment.pdf', 'coordinates': {'points': ((90.0, 204.3320000000001), (90.0, 240.3646), (434.01754999999997, 240.3646), (434.01754999999997, 204.3320000000001)), 'system': 'PixelSpace', 'layout_width':

In [ ]:

TYPE_WEIGHTS = {
    "Title": 1.0,
    "NarrativeText": 1.0,
    "ListItem": 0.9,
    "Table": 0.9,
    "Header": 0.8,
    "FigureCaption": 0.6,
    "UncategorizedText": 0.4,
}


def score(text, types):
    density = sum(c.isalnum() or c.isspace() for c in text) / max(len(text), 1)
    type_score = max((TYPE_WEIGHTS.get(t, 0.5) for t in types), default=0.5)
    return round((0.6 * density) + (0.4 * type_score), 3)


def chunk_elements(data, max_chars=2000, overlap=200, min_score=0.5):
    chunks = []
    buf = []
    types = set()
    current_heading = None
    page_start = None
    page_end = None
    need_overlap = False

    def flush():
        nonlocal buf, types
        nonlocal page_start, page_end
        nonlocal need_overlap
        if not buf:
            return
        text = " ".join(buf)
        text = re.sub(r"\s+", " ", text).strip()
        sc = score(text, types)
        if sc >= min_score:
            chunks.append({
                "id": hashlib.md5(text.encode()).hexdigest()[:8],
                "text": text,
                "page_start": page_start,
                "page_end": page_end,
                "heading": current_heading,
                "types": list(types),
                "score": sc,
            })
        # overlap only when chunk exceeded size
        if need_overlap and overlap > 0:
            overlap_text = text[-overlap:]
            # avoid cutting words
            split_idx = overlap_text.find(" ")
            if split_idx != -1:
                overlap_text = overlap_text[split_idx:].strip()
            buf = [overlap_text]
        else:
            buf = []
        types = set()
        page_start = None
        page_end = None
        need_overlap = False
        
    for el in data:
        text = el.get("text", "").strip()
        if not text:
            continue
        t = el.get("type", "Unknown")
        # update heading metadata only
        if t in ("Title", "Header"):
            current_heading = text
            # don't create chunk for title alone
            continue
        # keep tables separate
        if t == "Table":
            flush()
            table_text = re.sub(r"\s+", " ", text)
            chunks.append({
                "id": hashlib.md5(table_text.encode()).hexdigest()[:8],
                "text": table_text,
                "page_start": el.get("page"),
                "page_end": el.get("page"),
                "heading": current_heading,
                "types": ["Table"],
                "score": score(table_text, ["Table"]),
            })
            continue
        # page tracking
        if page_start is None:
            page_start = el.get("page")
        page_end = el.get("page")
        # chunk size check
        current_size = sum(len(x) for x in buf)
        if current_size + len(text) > max_chars:
            need_overlap = True
            flush()
            page_start = el.get("page")
            page_end = el.get("page")
        buf.append(text)
        types.add(t)
    flush()
    return chunks

In [40]:
chunked_data = chunk_elements(structured_data)

In [47]:
print("structured_data\n")
for i in structured_data:
    for key, items in i.items():
        print(key, " -> ", items,"\n")

structured_data

type  ->  Title 

text  ->  Centos OS deployment of Django 

page  ->  1 

filename  ->  centOSdepolyment.pdf 

coordinates  ->  {'points': ((90.0, 79.81145000000004), (90.0, 137.54144999999994), (351.06750000000005, 137.54144999999994), (351.06750000000005, 79.81145000000004)), 'system': 'PixelSpace', 'layout_width': 595.3, 'layout_height': 841.9} 

type  ->  Title 

text  ->  and Apache 

page  ->  1 

filename  ->  centOSdepolyment.pdf 

coordinates  ->  {'points': ((254.4, 79.81145000000004), (254.4, 105.86144999999999), (405.8615000000001, 105.86144999999999), (405.8615000000001, 79.81145000000004)), 'system': 'PixelSpace', 'layout_width': 595.3, 'layout_height': 841.9} 

type  ->  NarrativeText 

text  ->  Step-1 Update the system and install Apache and mod_wsgi: 

page  ->  1 

filename  ->  centOSdepolyment.pdf 

coordinates  ->  {'points': ((90.0, 204.3320000000001), (90.0, 240.3646), (434.01754999999997, 240.3646), (434.01754999999997, 204.3320000000001)), 's

In [44]:
print(chunked_data,"\n")
for i in chunked_data:
    for key, items in i.items():
        print(key, " -> ", items,"\n")

[{'id': 'fc57e78c', 'text': 'Step-1 Update the system and install Apache and mod_wsgi: Step-3 Create a virtual environment for django project python3 -m venv Step-4 Activate the virtual environment source environment_name/bin/activate Create a directory for the projects to be pasted Step-6 Configure Apache: Create a configuration file for your site: Config terminal opens there we should write the config Require all granted Require all granted </Files> </Directory> WSGIDaemonProcess python-path=/home/ WSGIProcessGroup poptest WSGIScriptAlias /home/popsite/poptestdir/poptest/wsgi.py </VirtualHost> Key points for writing config file DocumentRoot : it should be the path of the folder which contains manage.py file Alias /static : it should be the path of the folder <Directory /home/popsite/poptestdir/static>: : it should be the path of the folder <Directory /home/popsite/poptestdir/poptest>:it should be the path of the folder which contains wsgi.py file poptest:it should be the name of the 